## pricessing the CMORIZED stable runs

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
import matplotlib as mpl
import matplotlib.ticker as mticker
import netCDF4
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [3]:
from scipy import stats

In [4]:
mpl.rcParams['font.family'] = 'Droid Sans'
mpl.rcParams['font.size'] = 12
# Edit axes parameters
mpl.rcParams['axes.linewidth'] = 1.5
# Tick properties
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.minor.size'] = 3
mpl.rcParams['xtick.major.width'] = 1
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.minor.size'] = 3
mpl.rcParams['ytick.major.width'] = 1
mpl.rcParams['ytick.direction'] = 'out'

In [5]:
### Functions needed for the analysis

In [6]:
import matplotlib as m
from matplotlib.colors import BoundaryNorm as BM
import matplotlib.patches as mpatches

def plot_background(ax):
    ax.add_feature(cfeature.COASTLINE, alpha=0.9, lw=1.1)
    # ax.set_global()
    # ax.add_feature(cfeature.LAND, color='lightgray')
    # ax.add_feature(cfeature.OCEAN, color='lightgray')
    gl = ax.gridlines(draw_labels=True,
                      linewidth=1, color='gray', alpha=0.01, linestyle='--')
    gl.top_labels = False
    # gl.left_labels = False
    # gl.bottom_labels = False
    gl.right_labels = False
    gl.xlines = False
    # gl.xlocator = mticker.FixedLocator([-180, -45, 0, 45, 180])
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 10, 'color': 'k'}
    gl.ylabel_style = {'size': 10, 'color': 'k'}
    return ax


def plot_maps(x, y, z, titles, labels, cmap, levels, cbar_label = 'Precip', pval = [], nrows=1, ncols=3, figsize=(12,4), land_mask_list = [0], add_patch=False, cbar_orientation='vertical', hatch_type = 'insig'):
    fig, axarr = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, subplot_kw={'projection':ccrs.Robinson(central_longitude=180)})
    
    axlist = axarr.flatten()
    
    for ax in axlist:
        plot_background(ax)
    
    for i in range(len(z)):
        axlist[i].contourf(x, y, z[i], cmap = cmap, transform = ccrs.PlateCarree(central_longitude=0), levels=levels, extend='both')
        axlist[i].set_title(titles[i])
        if i in land_mask_list:
            axlist[i].add_feature(cfeature.LAND, color = 'k', zorder=1)
        if pval != []:
            if hatch_type == 'insig':
                pval_plot = np.ma.masked_less_equal(pval[i], 0.05)
            elif hatch_type == 'sig':
                pval_plot = np.ma.masked_greater(pval[i], 0.05)
            axlist[i].pcolor(x, y, pval_plot, alpha = 0., hatch='////', transform = ccrs.PlateCarree(central_longitude=0))
        axlist[i].set_title(titles[i], fontdict={'fontsize':12})
        axlist[i].text(0.1, 1.05, labels[i], size=16, fontweight='bold', transform=axlist[i].transAxes)
        if add_patch:
            axlist[i].add_patch(mpatches.Rectangle(xy=[120, -65], width=170, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[190, -5], width=80, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[140, -5], width=30, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[250, -30], width=40, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
        
    norm = BM(levels, 256, extend='both')
    fig.colorbar(m.cm.ScalarMappable(norm = norm, cmap=cmap), ax = axlist, \
                orientation = cbar_orientation, shrink=0.4, aspect = 20, pad = 0.05, label = cbar_label)

In [7]:
from functions import preproc_funcs as funcs

In [8]:
from functions import xr_lowess

In [9]:
import glob

In [10]:
files_pic = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i1p1f1/Amon/co2/gn/files/d20191115/*.nc'))
files_2030 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i1p1f2/Amon/co2/gn/files/d20250307/*.nc'))
files_2035 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i2p1f2/Amon/co2/gn/files/d20250307/*.nc'))
files_2040 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i3p1f2/Amon/co2/gn/files/d20250307/*.nc'))
files_2045 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/co2/gn/files/d20250307/*.nc'))
files_2050 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i5p1f2/Amon/co2/gn/files/d20250307/*.nc'))
files_2055 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i6p1f2/Amon/co2/gn/files/d20250307/*.nc'))
files_2060 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i7p1f2/Amon/co2/gn/files/d20250307/*.nc'))
files_2045

['/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/co2/gn/files/d20250307/co2_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_010101-011012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/co2/gn/files/d20250307/co2_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_011101-012012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/co2/gn/files/d20250307/co2_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_012101-013012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/co2/gn/files/d20250307/co2_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_013101-014012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/co2/gn/files/d20250307/co2_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_014101-015012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/co2/gn/files/d20250307/

In [11]:
ds2030 = xc.open_mfdataset(files_2030)
ds2035 = xc.open_mfdataset(files_2035)
ds2040 = xc.open_mfdataset(files_2040)
ds2045 = xc.open_mfdataset(files_2045)
ds2050 = xc.open_mfdataset(files_2050)
ds2055 = xc.open_mfdataset(files_2055)
ds2060 = xc.open_mfdataset(files_2060)

In [12]:
years = ['2030', '2035', '2040', '2045', '2050', '2055', '2060']
# years = ['2030', '2035', '2040', '2045', '2050', '2060']

In [13]:
collection = dict(zip(years, [ds2030, ds2035, ds2040, ds2045, ds2050, ds2055, ds2060]))
# collection = dict(zip(years, [ds2030, ds2035, ds2040, ds2045, ds2050, ds2060]))

In [14]:
# Function to process a single model and return the detrended NINO3.4 and precip anomalies
def process_model(model_identifier):
    try:
        # print(f"Processing model: {model_identifier}")
        # Load datasesiextents
        ds_stable = collection[model_identifier]
        # add custom time ranges
        stable_start_year = int(model_identifier)
        stable_end_year = int(stable_start_year) + 1000
        # var = ds_stable['fld_s16i222'].resample(time='AS-JUN').mean('time').load()
        var = ds_stable.co2.mean('plev').resample(time='AS-JUN').mean('time').load()
        var['time'] = xr.cftime_range(f'{stable_start_year}-01-01', f'{stable_end_year+1}-01-01', freq='1Y')
        #
        # precip = ds_precip['pr'] * 86400  # Convert kg/m²/s to mm/day

        # Calculate 3d values
        # var_anom = funcs.calc_anom(var, hist_access_r10.sel(time = slice('1960', '1990'))).resample(time = 'AS-JUN').mean('time').load()
        # siextents_trend = funcs.calc_trend3d(tos_anom.sel(time = slice('1980', '2014')), 'time')
        # siextents_trend_pval = funcs.calc_trend_pval3d(tos_anom.sel(time = slice('1980', '2014')), 'time')

        # calc timeseries values
        # weights = np.cos(np.deg2rad(siextents.lat))
        # gmst_anom = siextents_anom.weighted(weights).mean(('lat', 'lon'))
        # nino34_index = funcs.detrend1d_check(siextents_anom.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights).mean(('lat', 'lon')), period=15)
        # wp_sst = siextents_anom.sel(lat = slice(-5,5), lon = slice(140, 170)).weighted(weights).mean(('lat', 'lon'))
        # ct_sst = siextents_anom.sel(lat = slice(-5,5), lon = slice(190, 270)).weighted(weights).mean(('lat', 'lon'))
        # so_sst = siextents_anom.sel(lat = slice(-65, -45), lon = slice(120, 290)).weighted(weights).mean(('lat', 'lon'))

        # print(f'Completed: {model_identifie}')
        return model_identifier, var
    except Exception as e:
        print(f"Error processing {model_identifier}: {e}")
        return model_identifier



In [15]:
import xesmf as xe

In [16]:
import multiprocessing as mp

In [17]:
ds_out = xe.util.cf_grid_2d(-0.75, 360, 1.5, -90, 90, 1.5)
ds_out

<xarray.Dataset>
Dimensions:             (lon: 240, bound: 2, lat: 120)
Coordinates:
  * lon                 (lon) float64 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
  * lat                 (lat) float64 -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
    latitude_longitude  float64 nan
Dimensions without coordinates: bound
Data variables:
    lon_bounds          (lon, bound) float64 -0.75 0.75 0.75 ... 357.8 359.2
    lat_bounds          (lat, bound) float64 -90.0 -88.5 -88.5 ... 88.5 90.0

In [ ]:
# Run multiprocessing and gather results
res_arr = []
# with mp.Pool(processes=mp.cpu_count()) as pool:
with mp.Pool(processes=4) as pool:
    i = 0
    for res in pool.imap(process_model, years):
        res_arr.append(res)
        print(f'Completed {i+1}/{len(years)}', end='\r')
        i += 1



In [ ]:
variants = []
ds_arr = []
for i in range(7):
    variants.append(res_arr[i][0])
    ds_arr.append(res_arr[i][1])

In [ ]:
variants

['2030', '2035', '2040', '2045', '2050', '2055', '2060']

In [ ]:
ds_arr

[<xarray.DataArray 'ts' (time: 1001, lat: 145, lon: 192)>
 array([[[228.16125, 228.16125, 228.16125, ..., 228.16125, 228.16125,
          228.16125],
         [229.79301, 229.75435, 229.70801, ..., 229.91794, 229.88309,
          229.84158],
         [230.6716 , 230.55257, 230.4251 , ..., 231.0156 , 230.9067 ,
          230.78804],
         ...,
         [252.05244, 252.08005, 252.11504, ..., 251.97832, 252.01126,
          252.0206 ],
         [251.55095, 251.552  , 251.56259, ..., 251.51253, 251.52437,
          251.53915],
         [250.77364, 250.77364, 250.77364, ..., 250.77364, 250.77364,
          250.77364]],
 
        [[224.59491, 224.59491, 224.59491, ..., 224.59491, 224.59491,
          224.59491],
         [226.70683, 226.65654, 226.6028 , ..., 226.86653, 226.81136,
          226.75755],
         [227.68248, 227.55524, 227.43312, ..., 228.04144, 227.92256,
          227.80554],
 ...
         [258.21527, 258.26102, 258.32034, ..., 258.00177, 258.10245,
          258.14548],


In [ ]:
out = xr.concat([x.to_dataset(name='co2') for x in ds_arr], dim=variants).rename(dict(concat_dim = 'model'))

In [ ]:
regridder = xe.Regridder(out, ds_out, 'bilinear', periodic=True, ignore_degenerate=True)

In [ ]:
out_regrid = regridder(out)
out_regrid

<xarray.Dataset>
Dimensions:             (model: 6, time: 1026, lat: 120, lon: 240)
Coordinates:
  * time                (time) object 2030-12-31 00:00:00 ... 3055-12-31 00:0...
  * model               (model) object '2030' '2035' '2040' '2045' '2050' '2055'
    latitude_longitude  float64 nan
  * lon                 (lon) float64 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
  * lat                 (lat) float64 -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
Data variables:
    ts                  (model, time, lat, lon) float32 229.1 229.1 ... 269.9
Attributes:
    regrid_method:  bilinear

In [ ]:
out_regrid.to_netcdf('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/access_stable/access_stable_co2_original.nc')
# out.to_netcdf('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/access_stable_siextents_original.nc')

In [30]:
# out['siextents'].isel(model = 0).dropna('time').isel(time = slice(1, -1)).plot()

### analysing  the tcda

In [10]:
from numba import njit

def compute_thermocline_dataset(thetao, lev, max_depth=500, smooth_window=3):
    """
    Compute thermocline depth and temperature as a Dataset using Numba-accelerated gradient search.

    Parameters:
    -----------
    thetao : xarray.DataArray
        Potential temperature (time, lev, lat, lon)
    lev : xarray.DataArray
        Depth levels (1D, in meters)
    max_depth : float
        Maximum depth (in meters) to search for thermocline (default = 500 m)

    Returns:
    --------
    xarray.Dataset with:
        - thermocline_depth (time, lat, lon)
        - temp_at_thermocline (time, lat, lon)
    """

    # Ensure lev is increasing with depth
    if not np.all(np.diff(lev) > 0):
        lev = lev[::-1]
        thetao = thetao.sel(lev=lev)

    lev_vals = lev.values.astype(np.float32)

    # Smooth along vertical axis (lev)
    if smooth_window and smooth_window > 1:
        thetao = thetao.rolling(lev=smooth_window, center=True, min_periods=1).mean('lev')

    @njit
    def _find_thermocline(temp_prof, depth_prof, max_depth):
        n = len(temp_prof)
        max_grad = -1.0
        depth_at_max = np.nan
        temp_at_max = np.nan

        for i in range(n - 1):
            z1, z2 = depth_prof[i], depth_prof[i + 1]
            if z1 > max_depth or z2 > max_depth:
                continue

            t1, t2 = temp_prof[i], temp_prof[i + 1]
            if np.isnan(t1) or np.isnan(t2):
                continue

            dz = z2 - z1
            if dz <= 0:
                continue

            dTdz = abs((t2 - t1) / dz)
            if dTdz > max_grad:
                max_grad = dTdz
                depth_at_max = 0.5 * (z1 + z2)
                temp_at_max = 0.5 * (t1 + t2)

        return depth_at_max, temp_at_max

    def wrapper(temp_prof):
        return _find_thermocline(temp_prof, lev_vals, max_depth)

    depth, temp = xr.apply_ufunc(
        wrapper,
        thetao.chunk(dict(lev=-1, time=1, lat=64, lon=64)),
        input_core_dims=[["lev"]],
        output_core_dims=[[], []],
        vectorize=True,
        dask="parallelized",
        # dask="allowed",
        output_dtypes=[np.float32, np.float32],
    )

    return xr.Dataset({
        "thermocline_depth": depth.assign_attrs({
            "units": "m",
            "description": f"Depth of max |dT/dz| in upper {max_depth} m"
        }),
        "temp_at_thermocline": temp.assign_attrs({
            "units": thetao.attrs.get("units", "degC"),
            "description": "Temperature at thermocline depth"
        })
    })

In [11]:
def compute_isotherm_depth(thetao, lev, isotherm_temp=20.0, max_depth=500):
    """
    Compute the depth of a specified isotherm (e.g., 20°C) using linear interpolation.

    Parameters:
    -----------
    thetao : xarray.DataArray
        Potential temperature (time, lev, lat, lon)
    lev : xarray.DataArray
        Depth levels (in meters, 1D)
    isotherm_temp : float
        Target temperature for isotherm (default: 20°C)
    max_depth : float
        Maximum search depth (default: 500 m)

    Returns:
    --------
    isotherm_depth : xarray.DataArray (time, lat, lon)
        Depth at which temperature equals the specified isotherm
    """

    # Ensure increasing depth
    if not np.all(np.diff(lev) > 0):
        lev = lev[::-1]
        thetao = thetao.sel(lev=lev)

    lev_vals = lev.values.astype(np.float32)

    # @njit(parallel=False, nogil=True)
    @njit
    def _find_isotherm(temp_prof, depth_prof, T_iso, max_depth):
        n = len(temp_prof)
        for i in range(n - 1):
            z1, z2 = depth_prof[i], depth_prof[i + 1]
            if z1 > max_depth or z2 > max_depth:
                continue

            t1, t2 = temp_prof[i], temp_prof[i + 1]
            if np.isnan(t1) or np.isnan(t2):
                continue

            if (t1 - T_iso) * (t2 - T_iso) <= 0:
                # Crosses isotherm — linear interpolate
                if t2 != t1:
                    frac = (T_iso - t1) / (t2 - t1)
                    return z1 + frac * (z2 - z1)
                else:
                    return 0.5 * (z1 + z2)

        return np.nan

    def wrapper(temp_prof):
        return _find_isotherm(temp_prof, lev_vals, isotherm_temp, max_depth)

    isotherm_depth = xr.apply_ufunc(
        wrapper,
        thetao,
        input_core_dims=[["lev"]],
        output_core_dims=[[]],
        vectorize=True,
        dask="parallelized",
        # dask="allowed",
        output_dtypes=[np.float32],
    )

    isotherm_depth.name = f"isotherm_{int(isotherm_temp)}C_depth"
    isotherm_depth.attrs = {
        "units": "m",
        "description": f"Depth of {isotherm_temp}°C isotherm (upper {max_depth}m)"
    }

    return isotherm_depth

In [12]:
files_pic = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i1p1f1/Omon/thetao/gn/files/d20191115/*.nc'))
files_2030 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i1p1f2/Omon/thetao/gn/files/d20250307/*.nc'))
files_2035 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i2p1f2/Omon/thetao/gn/files/d20250307/*.nc'))
files_2040 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i3p1f2/Omon/thetao/gn/files/d20250307/*.nc'))
files_2045 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Omon/thetao/gn/files/d20250307/*.nc'))
files_2050 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i5p1f2/Omon/thetao/gn/files/d20250307/*.nc'))
files_2055 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i6p1f2/Omon/thetao/gn/files/d20250307/*.nc'))
files_2060 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i7p1f2/Omon/thetao/gn/files/d20250307/*.nc'))
files_2045

['/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Omon/thetao/gn/files/d20250307/thetao_Omon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_010101-011012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Omon/thetao/gn/files/d20250307/thetao_Omon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_011101-012012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Omon/thetao/gn/files/d20250307/thetao_Omon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_012101-013012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Omon/thetao/gn/files/d20250307/thetao_Omon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_013101-014012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Omon/thetao/gn/files/d20250307/thetao_Omon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_014101-015012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f

In [13]:
test = xr.open_mfdataset(files_pic[0]).thetao
test

<xarray.DataArray 'thetao' (time: 120, lev: 50, j: 300, i: 360)>
dask.array<open_dataset-4fd6019f90a437ba8b242bcedb950055thetao, shape=(120, 50, 300, 360), dtype=float32, chunksize=(120, 50, 300, 360), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) object 0271-01-16 12:00:00 ... 0280-12-16 12:00:00
  * lev        (lev) float64 5.0 15.0 25.0 ... 5.166e+03 5.499e+03 5.831e+03
  * j          (j) int32 0 1 2 3 4 5 6 7 8 ... 292 293 294 295 296 297 298 299
  * i          (i) int32 0 1 2 3 4 5 6 7 8 ... 352 353 354 355 356 357 358 359
    latitude   (j, i) float64 dask.array<chunksize=(300, 360), meta=np.ndarray>
    longitude  (j, i) float64 dask.array<chunksize=(300, 360), meta=np.ndarray>
Attributes:
    standard_name:  sea_water_potential_temperature
    long_name:      Sea Water Potential Temperature
    comment:        Diagnostic should be contributed even for models using co...
    units:          degC
    cell_methods:   area: mean where sea time: mean
    cell_measures:  area: areacello volume: volcello
    history:        2019-11-15T17:36:24Z altered by CMOR: replaced missing va...

In [14]:
# Assume your coordinate variables are named 'lat' and 'lon' and depend on (i, j)
lat = test['latitude']  # shape: (i, j)
lon = test['longitude']

# Define your region
lat_min, lat_max = -25, 25
lon_min, lon_max = 100, 300

# Build a mask for where lat/lon fall within the box
mask = ((lat >= lat_min) & (lat <= lat_max) &
        (lon >= lon_min) & (lon <= lon_max))

In [15]:
mask_loaded = mask.load()

In [16]:
ds2030 = xc.open_mfdataset(files_2030, parallel=True).thetao.sel(lev = slice(0, 400)).chunk(dict(time=12, lev=-1, i=64, j=64)).where(mask_loaded, drop=True)
ds2035 = xc.open_mfdataset(files_2035, parallel=True).thetao.sel(lev = slice(0, 400)).chunk(dict(time=12, lev=-1, i=64, j=64)).where(mask_loaded, drop=True)
ds2040 = xc.open_mfdataset(files_2040, parallel=True).thetao.sel(lev = slice(0, 400)).chunk(dict(time=12, lev=-1, i=64, j=64)).where(mask_loaded, drop=True)
ds2045 = xc.open_mfdataset(files_2045, parallel=True).thetao.sel(lev = slice(0, 400)).chunk(dict(time=12, lev=-1, i=64, j=64)).where(mask_loaded, drop=True)
ds2050 = xc.open_mfdataset(files_2050, parallel=True).thetao.sel(lev = slice(0, 400)).chunk(dict(time=12, lev=-1, i=64, j=64)).where(mask_loaded, drop=True)
ds2055 = xc.open_mfdataset(files_2055, parallel=True).thetao.sel(lev = slice(0, 400)).chunk(dict(time=12, lev=-1, i=64, j=64)).where(mask_loaded, drop=True)
ds2060 = xc.open_mfdataset(files_2060, parallel=True).thetao.sel(lev = slice(0, 400)).chunk(dict(time=12, lev=-1, i=64, j=64)).where(mask_loaded, drop=True)

In [17]:
years = ['2030', '2035', '2040', '2045', '2050', '2055', '2060']

In [18]:
collection = dict(zip(years, [ds2030, ds2035, ds2040, ds2045, ds2050, ds2055, ds2060]))

In [19]:
from dask.diagnostics import ProgressBar    

In [20]:
# Function to process a single model and return the detrended NINO3.4 and precip anomalies
def process_model_for_thetao(model_identifier):
    try:
        # print(f"Processing model: {model_identifier}")
        # Load datasetos
        ds_stable = collection[model_identifier]
        # add custom time ranges
        stable_start_year = int(model_identifier)
        stable_end_year = int(stable_start_year) + 1000
        # var = ds_stable['fld_s16i222'].resample(time='AS-JUN').mean('time').load()
        var = ds_stable.resample(time='AS-JUN').mean('time')
        var['time'] = xr.cftime_range(f'{stable_start_year}-01-01', f'{stable_end_year+1}-01-01', freq='1Y')
        with ProgressBar():
            t20d = (compute_isotherm_depth(var, var.lev)).load()
        #
        # precip = ds_precip['pr'] * 86400  # Convert kg/m²/s to mm/day

        # Calculate 3d values
        # var_anom = funcs.calc_anom(var, hist_access_r10.sel(time = slice('1960', '1990'))).resample(time = 'AS-JUN').mean('time').load()
        # tos_trend = funcs.calc_trend3d(tos_anom.sel(time = slice('1980', '2014')), 'time')
        # tos_trend_pval = funcs.calc_trend_pval3d(tos_anom.sel(time = slice('1980', '2014')), 'time')

        # calc timeseries values
        # weights = np.cos(np.deg2rad(tos.lat))
        # gmst_anom = tos_anom.weighted(weights).mean(('lat', 'lon'))
        # nino34_index = funcs.detrend1d_check(tos_anom.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights).mean(('lat', 'lon')), period=15)
        # wp_sst = tos_anom.sel(lat = slice(-5,5), lon = slice(140, 170)).weighted(weights).mean(('lat', 'lon'))
        # ct_sst = tos_anom.sel(lat = slice(-5,5), lon = slice(190, 270)).weighted(weights).mean(('lat', 'lon'))
        # so_sst = tos_anom.sel(lat = slice(-65, -45), lon = slice(120, 290)).weighted(weights).mean(('lat', 'lon'))

        # print(f'Completed: {model_identifie}')
        return model_identifier, t20d
    except Exception as e:
        print(f"Error processing {model_identifier}: {e}")
        return model_identifier



In [21]:
res_arr = []
for i in range(len(years)):
    res = process_model_for_thetao(years[i])
    res_arr.append(res)

[########################################] | 100% Completed | 31m 11s
[########################################] | 100% Completed | 32m 42s
[########################################] | 100% Completed | 33m 17s
[########################################] | 100% Completed | 30m 34s
[########################################] | 100% Completed | 32m 47s
[########################################] | 100% Completed | 32m 42s
[########################################] | 100% Completed | 31m 39s


In [22]:
model_list = np.array(res_arr)[:, 0]
model_list

array(['2030', '2035', '2040', '2045', '2050', '2055', '2060'],
      dtype=object)

In [23]:
model_var = xr.concat(np.array(res_arr)[:, 1], dim=model_list, coords='minimal', compat='override').rename(dict(concat_dim = 'model')).to_dataset(name = 'tcda')

In [24]:
out = xr.merge([model_var])
out

<xarray.Dataset>
Dimensions:    (j: 110, i: 200, time: 1031, model: 7)
Coordinates:
  * j          (j) int32 82 83 84 85 86 87 88 89 ... 185 186 187 188 189 190 191
  * i          (i) int32 20 21 22 23 24 25 26 27 ... 213 214 215 216 217 218 219
  * time       (time) object 2030-12-31 00:00:00 ... 3060-12-31 00:00:00
    latitude   (j, i) float64 -24.6 -24.6 -24.6 -24.6 ... 24.6 24.6 24.6 24.6
    longitude  (j, i) float64 100.5 101.5 102.5 103.5 ... 297.5 298.5 299.5
  * model      (model) object '2030' '2035' '2040' '2045' '2050' '2055' '2060'
Data variables:
    tcda       (model, time, j, i) float32 197.3 175.8 171.6 ... nan nan nan

In [25]:
out.to_netcdf('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/access_stable_tcda_original.nc')